# MoodNote-AI — Synthetic diary data generation (phase 3, Research Content 1)

Runs on a **Colab T4 GPU**: Llama-3.1-8B-Instruct and Qwen3-8B (HF transformers, 4-bit) generate diary
entries, then the pool is deduplicated and checked for leakage, cross-LLM audited, and blind manual-audit
sheets are exported.

Before you start:
- Runtime → Change runtime type → **T4 GPU**.
- Colab Secrets: `HF_TOKEN` — from an HF account that has accepted the `meta-llama/Llama-3.1-8B-Instruct` license.
- Edit prompts/config locally and push to branch `BRANCH`; Cell 1 runs `git pull`.

Output lives on Drive at `MyDrive/MoodNote-AI/synthetic/` (symlinked to `data/synthetic`). If the runtime
disconnects, re-run the same cell: every step resumes and only does the missing work.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os
import subprocess

import torch
from google.colab import drive, userdata

BRANCH = "feature/ToanHuynh/NCKH-rebuild"
REPO_DIR = "/content/MoodNote-AI"
DRIVE_DIR = "/content/drive/MyDrive/MoodNote-AI/synthetic"

if not torch.cuda.is_available():
    raise RuntimeError("GPU not found. Runtime -> Change runtime type -> T4 GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}")

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(
        ["git", "clone", "-b", BRANCH, "https://github.com/MoodNote/MoodNote-AI.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)

In [ ]:
# ── Cell 2: Dependencies + HF login ──────────────────────────────────────────
# transformers/accelerate match requirements.txt; bitsandbytes is needed for 4-bit.
!pip install -q transformers==5.3.0 accelerate==1.13.0 bitsandbytes==0.50.2 rapidfuzz==3.14.5

from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
# ── Cell 3: Data dirs + UIT-VSMEC (the leakage guard needs data/real/raw) ───
os.makedirs("data", exist_ok=True)
if not os.path.islink("data/synthetic"):
    os.symlink(DRIVE_DIR, "data/synthetic")

!python -m src.data.real.download_vsmec

## Prompt trial (small scale)

2 samples per label per model into `data/synthetic_trial` (Colab local disk, not Drive). Read them: do
they sound like a personal diary, or do they drift into social-media style or read as formulaic? To
revise, edit `src/data/synthetic/prompts.py` (add a new template id) → push → re-run Cell 1 and this cell.

In [ ]:
# ── Cell 4: Prompt trial ─────────────────────────────────────────────────────
import pandas as pd

!rm -rf data/synthetic_trial
!python -m src.data.synthetic.generate --model llama --per-label 2 --data-dir data/synthetic_trial
!python -m src.data.synthetic.generate --model qwen --per-label 2 --data-dir data/synthetic_trial

pd.set_option("display.max_colwidth", None)
trial = pd.concat(
    pd.read_json(f"data/synthetic_trial/raw/{m}.jsonl", lines=True, convert_dates=False)
    for m in ("llama", "qwen")
)
trial[["id", "van_phong", "do_dai", "ngu_canh", "truncated", "text"]]

## Bulk generation + filtering + cross-LLM audit

Each model generates 200 samples per label (1,400 samples). If a label ends up short after filtering,
re-run generate with a larger `--per-label` (it only tops up the missing rows), then re-run filter.

In [ ]:
# ── Cell 5: Generate (Llama) ──────────────────────────────────────────────────
!python -m src.data.synthetic.generate --model llama

In [ ]:
# ── Cell 6: Generate (Qwen) ───────────────────────────────────────────────────
!python -m src.data.synthetic.generate --model qwen

In [ ]:
# ── Cell 7: Filter (empty/truncated, dedup, leakage vs VSMEC) ───────────────
!python -m src.data.synthetic.filter

In [ ]:
# ── Cell 8: Cross-LLM audit (Qwen audits Llama, then Llama audits Qwen) ─────
!python -m src.qa.cross_llm_audit --auditor qwen
!python -m src.qa.cross_llm_audit --auditor llama

In [ ]:
# ── Cell 9: Export blind manual-audit sheets ─────────────────────────────────
!python -m src.qa.manual_audit

## Next steps (local machine)

1. Download `MyDrive/MoodNote-AI/synthetic/` into `data/synthetic/`.
2. The author and the collaborator fill the `label` column of `audit/rater_a.csv` / `audit/rater_b.csv`
   **independently** (one of 7 labels: Enjoyment, Sadness, Anger, Fear, Disgust, Surprise, Other). Do not
   edit, delete or reorder rows; save as **CSV UTF-8**.
3. `python -m src.qa.acceptance_gate` → `reports/qa_report_<template>.json`; on pass it writes
   `data/synthetic/accepted/`.